# Rust Jupyter environment — smoke test

This notebook exercises each pre-warmed crate so you can confirm the image
is healthy. Because these crates are compiled into the image cache at build
time, the `:dep` lines below resolve almost instantly instead of triggering
a multi-minute compile.

Standard crate set: `ndarray`, `polars`, `linfa`, `linfa-clustering`,
`linfa-trees`, `smartcore`, `plotters`, `argmin`.

In [2]:
:dep ndarray = { version = "0.15" }
use ndarray::array;
let a = array![[1.0_f64, 2.0], [3.0, 4.0]];
println!("matrix:\n{}", a);
println!("sum = {}", a.sum());

matrix:
[[1, 2],
 [3, 4]]
sum = 10


In [3]:
:dep polars = { version = "0.44", features = ["lazy"] }
use polars::prelude::*;
let df = df![
    "name"  => ["a", "b", "c"],
    "value" => [1.0_f64, 2.0, 3.0],
]?;
println!("{}", df);

The type of the variable a was redefined, so was lost.


shape: (3, 2)
┌──────┬───────┐
│ name ┆ value │
│ ---  ┆ ---   │
│ str  ┆ f64   │
╞══════╪═══════╡
│ a    ┆ 1.0   │
│ b    ┆ 2.0   │
│ c    ┆ 3.0   │
└──────┴───────┘


In [4]:
:dep plotters = { version = "0.3", default-features = false, features = ["evcxr", "all_series", "all_elements"] }
use plotters::prelude::*;
evcxr_figure((360, 240), |root| {
    root.fill(&WHITE)?;
    let mut chart = ChartBuilder::on(&root)
        .caption("y = x^2", ("sans-serif", 18))
        .margin(10)
        .x_label_area_size(30)
        .y_label_area_size(30)
        .build_cartesian_2d(-3f32..3f32, 0f32..9f32)?;
    chart.configure_mesh().draw()?;
    chart.draw_series(LineSeries::new(
        (-30..=30).map(|x| x as f32 / 10.0).map(|x| (x, x * x)),
        &RED,
    ))?;
    Ok(())
})

y = x^2
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
0.0
 
 
 
1.0
 
 
 
2.0
 
 
 
3.0
 
 
 
4.0
 
 
 
5.0
 
 
 
6.0
 
 
 
7.0
 
 
 
8.0
 
 
 
9.0
 
 
 
 
-3.0
 
 
 
-2.0
 
 
 
-1.0
 
 
 
0.0
 
 
 
1.0
 
 
 
2.0
 
 
 
3.0
 
 
<polyline fill="none" opacity="1" stroke="#FF0000" stroke-width="1" points="40,35 45,46 50,57 55,67 60,76 65,86 70,95 76,103 81,111 86,119 91,127 96,134 101,140 106,147 112,153 117,158 122,164 127,169 132,173 137,177 143,181 148,185 153,188 158,191 163,193 168,195 173,197 179,198 184,199 189,199 194,199 199,199 204,199 209,198 215,197 220,195 225,193 230,191 235,188 240,185 246,181 251,177 256,173 261,169 266,164 271,158 276,153 282,147 287,140 292,134 297,127 302,119 307,111 312,103 318,95 323,86 328,76 333,67 338,57 343,46 349,35 "/>

In [5]:
:dep linfa = { version = "0.7" }
:dep linfa-clustering = { version = "0.7" }
// ndarray 0.15 was already pulled above to match linfa's version.
use linfa::prelude::*;
use linfa::DatasetBase;
use linfa_clustering::KMeans;
use ndarray::array;

// Wrapped in a block: evcxr can't name KMeans' opaque model type to persist it
// across cells, so we keep the model local to a single compiled statement.
{
    let data = array![[1.0_f64, 1.0], [1.2, 0.9], [5.0, 5.1], [5.2, 4.8]];
    let dataset = DatasetBase::from(data);
    let model = KMeans::params(2)
        .max_n_iterations(100)
        .fit(&dataset)
        .expect("KMeans fit failed");
    let assignments = model.predict(&dataset);
    println!("cluster assignments: {:?}", assignments);
}

cluster assignments: [0, 0, 1, 1], shape=[4], strides=[1], layout=CFcf (0xf), const ndim=1


()

In [6]:
:dep smartcore = { version = "0.3" }
// Confirm the remaining pre-warmed crates link cleanly.
:dep linfa-trees = { version = "0.7" }
:dep argmin = { version = "0.10" }
println!("smartcore, linfa-trees, argmin all available");

smartcore, linfa-trees, argmin all available


## Exporting

From the host:

```bash
make export-html   NOTEBOOK=example.ipynb
make export-script NOTEBOOK=example.ipynb
make export-pdf    NOTEBOOK=example.ipynb   # only if built with `make build-pdf`
make pair          NOTEBOOK=example.ipynb   # create a git-friendly .md twin
```

The `.ipynb` file itself is already the most portable artifact — it opens on
any machine with Jupyter + the evcxr kernel, or back in this same container.